# Task 3.2 — Advanced Computer Vision Project
## CIFAR-10 Image Classification with TensorFlow/Keras

A real-world-style computer vision classification workflow using the CIFAR-10 dataset:
data loading → preprocessing → CNN → training → evaluation → confusion matrix → sample predictions → saved model.

The model metrics are produced by execution; they are not hard-coded.


In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

print("TensorFlow version:", tf.__version__)


In [ ]:
# CIFAR-10 is a real, standard image classification dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
y_train = y_train.flatten()
y_test = y_test.flatten()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Train:", x_train.shape)
print("Test :", x_test.shape)


In [ ]:
# Sample images
plt.figure(figsize=(10, 5))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# CNN model
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:
# Train
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=2, restore_best_weights=True
    )
]

history = model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=10,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)


In [ ]:
# Evaluate and save
test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")

model.save("Task_3.2_CIFAR10_TensorFlow_CNN.keras")
print("Saved: Task_3.2_CIFAR10_TensorFlow_CNN.keras")


In [ ]:
# Curves
plt.figure(figsize=(7, 4))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CIFAR-10 CNN Accuracy")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CIFAR-10 CNN Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrix and classification report
y_prob = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_prob, axis=1)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(9, 9))
disp.plot(ax=ax, xticks_rotation=45, cmap="Blues", colorbar=False)
plt.title("CIFAR-10 Confusion Matrix")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names))


In [ ]:
# Prediction visualization
plt.figure(figsize=(10, 7))
for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(x_test[i])
    pred_name = class_names[y_pred[i]]
    true_name = class_names[y_test[i]]
    plt.title(f"True: {true_name}\nPred: {pred_name}")
    plt.axis("off")
plt.tight_layout()
plt.show()


## Project Summary

**Problem:** classify 32×32 RGB images into one of 10 CIFAR-10 categories.

**Approach:** normalize images, train a convolutional neural network with convolution, batch normalization, pooling, dropout, and dense layers.

**Evaluation:** test accuracy/loss, confusion matrix, classification report, and prediction visualization.

**Deliverable:** the notebook plus the generated `.keras` model file after execution.
